# 05-2. 분리·검색·치환 — 풀이 검증

## Goal

구분자 수와 메시지 내 구분자 경계를 검증한다.

> 학습자용 TODO를 먼저 완성한 뒤 참고한다.


## Setup

fixture와 실행 환경을 확인한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


fixture_path = FIXTURE_DIR / "delimited-events.txt"
lines = fixture_path.read_text(encoding="utf-8").splitlines()


## Steps

참고 구현을 실행한다.


In [ ]:
SENSITIVE_KEYS = {"password", "token", "secret"}


class EventParseError(ValueError):
    def __init__(self, code: str, message: str) -> None:
        super().__init__(message)
        self.code = code


def parse_event(line: str) -> dict:
    parts = line.split("|", maxsplit=2)
    if len(parts) != 3:
        raise EventParseError("FIELD_COUNT", "필드는 세 개여야 한다.")
    date, level, message = (part.strip() for part in parts)
    if not all((date, level, message)):
        raise EventParseError("EMPTY_FIELD", "빈 필드를 허용하지 않는다.")
    return {"date": date, "level": level, "message": message, "raw": line}


def mask_pairs(text: str) -> str:
    masked_parts = []
    for part in text.split("&"):
        key, separator, value = part.partition("=")
        if separator and key.casefold() in SENSITIVE_KEYS:
            value = "***"
        masked_parts.append(key + separator + value)
    return "&".join(masked_parts)


parsed, errors = [], []
for line_number, line in enumerate(lines, start=1):
    try:
        parsed.append(parse_event(line))
    except EventParseError as exc:
        errors.append({
            "line": line_number,
            "raw": line,
            "code": exc.code,
            "message": str(exc),
        })

masked_messages = [mask_pairs(record["message"]) for record in parsed]
print("정상:", len(parsed), "오류:", len(errors))
print("보고서용 메시지:", masked_messages)


## Checks

경계값과 fixture 결과를 대조한다.


In [ ]:
assert parse_event("2026-08-14 | INFO | a | b")["message"] == "a | b"
assert len(parsed) == 3 and len(errors) == 1
assert errors[0]["line"] == 4 and errors[0]["code"] == "FIELD_COUNT"
assert errors[0]["raw"] == lines[3]
try:
    parse_event("2026-08-14 | | empty")
except EventParseError as exc:
    assert exc.code == "EMPTY_FIELD"
else:
    raise AssertionError("빈 필드를 허용했다.")
assert all("training-token" not in message for message in masked_messages)
print("검증 통과")


## Next Steps

오류 원문의 저장 여부는 민감정보 정책과 재처리 필요성을 함께 고려한다.
